### Fase 1: Análise Exploratória de Dados (EDA)

In [0]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier 
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("/Workspace/Users/leonardobadell@protonmail.com/Projeto-Avaliativo---MODULO-1/E Commerce Dataset.xlsx - E Comm.csv")
print("E Commerce Dataset carregado com sucesso!\n")


# Primeiras linhas do dataset
print("Visualização das primeiras linhas do dataset\n")
display(df.head(10))

# Remover coluna CustomerID (não é relevante para análise)
df = df.drop(columns=['CustomerID'])
print("\nColuna 'CustomerID' removida do dataset (sem valor preditivo)\n")


# Valores estatísticos totais do dataset
print("Cálculo estatístico total:\n")
display(df.describe())


# Tipo de dados por coluna
print("Tipo de dados por coluna do dataset:\n")
df.info()

print("=" * 50)
print("Verificando se há valores nulos:")
print("=" * 50)
display(df.isnull().sum())


In [0]:
# ==================== ANÁLISE DE DESBALANCEAMENTO DOS DADOS ====================
print("="*50)
print("ANÁLISE EXPLORATÓRIA DE DADOS - CHURN")
print("="*50)

# Contando quantos clientes fizeram churn e quantos não fizeram
churn_counts = df['Churn'].value_counts()
total_clientes = len(df)

print(f"\nTotal de registros: {total_clientes}")
print(f"Classe 0 (Não Churn): {churn_counts[0]} ({churn_counts[0]/total_clientes*100:.2f}%)")
print(f"Classe 1 (Churn): {churn_counts[1]} ({churn_counts[1]/total_clientes*100:.2f}%)")
print(f"Razão de desbalanceamento: {churn_counts[0]/churn_counts[1]:.2f}:1")
print("\nCONCLUSÃO: Dataset está DESBALANCEADO!")
print("="*50)

# ==================== GRÁFICO 1: BARRAS COM TOTAIS ====================
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(['Não Churn (0)', 'Churn (1)'], 
              [churn_counts[0], churn_counts[1]], 
              color=['green', 'red'], 
              edgecolor='black')

ax.set_ylabel('Quantidade de Clientes', fontsize=12)
ax.set_title('Gráfico 1: Distribuição de Classes - Churn', fontsize=14, fontweight='bold')

# Adicionando valores totais nas barras
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height,
            f'{int(height)}',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# ==================== GRÁFICO 2: VIOLIN PLOT  ====================
print("\n" + "="*50)
print("GRÁFICO 2: VIOLIN PLOT - Distribuição de Variáveis")
print("="*50)

# Definir colunas relevantes
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
colunas = ['Tenure', 'SatisfactionScore', 'DaySinceLastOrder', 
           'WarehouseToHome', 'CashbackAmount', 'Complain']

# Criar gráfico de violino para cada coluna
for i, col in enumerate(colunas):
    sns.violinplot(data=df, x='Churn', y=col, hue='Churn', ax=axes[i],
                   palette={0: '#2ecc71', 1: '#e74c3c'}, 
                   inner='quartile', legend=False)
    # Configurar labels manualmente para evitar warnings
    axes[i].set_xticks([0, 1])
    axes[i].set_xticklabels(['Retido (0)', 'Churn (1)'])
    axes[i].set_title(f'{col} vs Churn', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Status do Cliente', fontsize=10)
    axes[i].set_ylabel(col, fontsize=10)

plt.suptitle('Violin Plots: Densidade das Distribuições vs Churn', 
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print("\n✅ Violin plots gerados com sucesso!")
print("="*50)


# ==================== GRÁFICO 3: HEATMAP DE CORRELAÇÃO ====================
print("\n" + "="*50)
print("GRÁFICO 3: HEATMAP - Correlação de Pearson")
print("="*50)

# Selecionar colunas numéricas relevantes
colunas_heatmap = [
    'Churn',
    'Tenure',
    'Complain',
    'SatisfactionScore',
    'DaySinceLastOrder',
    'OrderCount',
    'CouponUsed',
    'WarehouseToHome',
    'HourSpendOnApp',
    'CashbackAmount',
    'OrderAmountHikeFromlastYear'
]

# Calcular matriz de correlação
correlacao = df[colunas_heatmap].corr()

# Criar heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(correlacao, 
            annot=True, 
            fmt='.2f', 
            cmap='RdYlBu_r', 
            center=0,
            square=True,
            linewidths=0.5)
plt.title('Correlação de Pearson - Fatores que Afetam Churn', 
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("Correlações com Churn (ordenadas):")
print("="*70)
churn_corr = correlacao['Churn'].sort_values(ascending=False)
print(churn_corr)
print("="*70)

%md
## 📊 Análise Exploratória de Dados (EDA)

### 🎯 Objetivo
Identificar padrões e fatores que influenciam o **churn de clientes** no dataset de E-Commerce.

---

## 📈 Principais Descobertas

### **1. Desbalanceamento de Classes**
* Dataset fortemente **desbalanceado** (4.94:1)
  - Não Churn: 4.682 (83.16%)
  - Churn: 948 (16.84%)
* **Ação:** Aplicar técnicas de balanceamento (SMOTE, class weights) na modelagem

---

### **2. Correlações com Churn**

**Fortes (|r| > 0.30):**
* **Tenure (-0.35):** Variável mais preditiva — clientes novos têm maior risco

**Moderadas (0.15 < |r| < 0.30):**
* **Complain (+0.25):** Reclamações aumentam drasticamente o risco
* **DaySinceLastOrder (-0.16):** Padrões de compra recente importam
* **CashbackAmount (-0.15):** Cashback reduz churn

**Irrelevantes (|r| < 0.10):**
* `HourSpendOnApp`, `CouponUsed`, `OrderAmountHikeFromlastYear`, `OrderCount`

---

### **3. Qualidade dos Dados**
* **Valores nulos:** 7 colunas com 4-6% de missings
* **Duplicatas:** 556 linhas duplicadas identificadas
* **Outliers:** Detectados, mas mantidos (representam comportamentos legítimos)

---

### Fase 2: Tratamento e Limpeza (Data Prep)

In [0]:
# ==================== TRATAMENTO DE DUPLICATAS ====================
print("=" * 70)
print("TRATAMENTO DE DADOS: Identificação e Remoção de Duplicatas")
print("=" * 70)

# Verificar quantidade de duplicatas
num_duplicatas = df.duplicated().sum()
print(f"\nLinhas duplicadas encontradas: {num_duplicatas}")

if num_duplicatas > 0:
    print(f"Tamanho do dataset ANTES da remoção: {len(df)} linhas")
    
    # Remover duplicatas (mantém a primeira ocorrência)
    df = df.drop_duplicates()
    
    print(f"Tamanho do dataset APÓS a remoção: {len(df)} linhas")
    print(f"✅ {num_duplicatas} linha(s) duplicada(s) removida(s) com sucesso!")
else:
    print("✅ Nenhuma duplicata encontrada! Dataset limpo.")

print("=" * 70)

In [0]:
# ==================== TRATAMENTO DE VALORES NULOS ====================
print("\n" + "=" * 70)
print("TRATAMENTO DE DADOS: Preenchimento de Valores Nulos")
print("=" * 70)

# Verificar valores nulos antes do tratamento
print("\nValores nulos ANTES do tratamento:")
print(df.isnull().sum())

# Tratamento de valores nulos em variáveis numéricas
print("\n" + "-" * 70)
print("Aplicando estratégias de preenchimento...")
print("-" * 70)

# MEDIANA: Mais robusta contra outliers (para variáveis com valores extremos)
df['Tenure'] = df['Tenure'].fillna(df['Tenure'].median())
print("\u2705 'Tenure' preenchido com MEDIANA (resistente a outliers)")

df['DaySinceLastOrder'] = df['DaySinceLastOrder'].fillna(df['DaySinceLastOrder'].median())
print("\u2705 'DaySinceLastOrder' preenchido com MEDIANA")

df['WarehouseToHome'] = df['WarehouseToHome'].fillna(df['WarehouseToHome'].median())
print("\u2705 'WarehouseToHome' preenchido com MEDIANA")

df['CouponUsed'] = df['CouponUsed'].fillna(df['CouponUsed'].median())
print("\u2705 'CouponUsed' preenchido com MEDIANA (variável discreta com assimetria)")

df['OrderCount'] = df['OrderCount'].fillna(df['OrderCount'].median())
print("\u2705 'OrderCount' preenchido com MEDIANA (variável discreta com assimetria)")

# MÉDIA: Para variáveis com distribuição mais uniforme
df['HourSpendOnApp'] = df['HourSpendOnApp'].fillna(df['HourSpendOnApp'].mean())
print("\u2705 'HourSpendOnApp' preenchido com MÉDIA")

df['OrderAmountHikeFromlastYear'] = df['OrderAmountHikeFromlastYear'].fillna(df['OrderAmountHikeFromlastYear'].mean())
print("\u2705 'OrderAmountHikeFromlastYear' preenchido com MÉDIA")

# Verificar valores nulos após o tratamento
print("\n" + "-" * 70)
print("Valores nulos APÓS o tratamento:")
print("-" * 70)
print(df.isnull().sum())

total_nulos_restantes = df.isnull().sum().sum()
if total_nulos_restantes == 0:
    print("\n\u2705 SUCESSO! Todos os valores nulos foram tratados!")
else:
    print(f"\n\u26a0\ufe0f ATEN\u00c7\u00c3O: Ainda restam {total_nulos_restantes} valores nulos em outras colunas.")

print("=" * 70)
print(f"Dataset final limpo: {len(df)} linhas e {len(df.columns)} colunas")
print("=" * 70)

In [0]:
# ==================== BOXPLOT - DETECÇÃO DE OUTLIERS ====================
print("\n" + "="*70)
print("BOXPLOT - Distribuição de Variáveis por Churn")
print("="*70)

# Variáveis para boxplot
variaveis_boxplot = ['Tenure', 'SatisfactionScore', 'DaySinceLastOrder', 
                     'WarehouseToHome', 'CashbackAmount']

# Criar figura com 2 linhas x 3 colunas
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

# Cores personalizadas: verde para Retido (0), vermelho para Churn (1)
cores = ['#2ecc71', '#e74c3c']

# Criar boxplots para as 5 primeiras variáveis
for i, var in enumerate(variaveis_boxplot):
    # Calcular medianas para cada grupo
    med0 = df[df['Churn']==0][var].median()
    med1 = df[df['Churn']==1][var].median()
    
    # Criar boxplot
    bp = axes[i].boxplot([df[df['Churn']==0][var].dropna(), 
                          df[df['Churn']==1][var].dropna()],
                         tick_labels=['Retido (0)', 'Churn (1)'],
                         patch_artist=True,
                         widths=0.6)
    
    # Aplicar cores
    for patch, color in zip(bp['boxes'], cores):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    # Configurar título e labels
    axes[i].set_title(f'Distribuição: {var} vs Churn', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Status do Cliente', fontsize=10)
    axes[i].set_ylabel(var, fontsize=10)
    
    # Adicionar texto com medianas
    axes[i].text(0.5, 0.98, f'Med(0)={med0:.1f} | Med(1)={med1:.1f}', 
                transform=axes[i].transAxes, fontsize=9,
                verticalalignment='top', horizontalalignment='center',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# GRÁFICO 6: Distribuição de Reclamações vs Churn (gráfico de barras)
reclamacao_churn = pd.crosstab(df['Complain'], df['Churn'])

# Posições das barras
x_pos = [0, 1]
width = 0.35

# Criar gráfico de barras agrupadas
axes[5].bar([p - width/2 for p in x_pos], 
            reclamacao_churn[0], 
            width, 
            label='Retido (0)', 
            color='#2ecc71', 
            edgecolor='black')

axes[5].bar([p + width/2 for p in x_pos], 
            reclamacao_churn[1], 
            width, 
            label='Churn (1)', 
            color='#e74c3c', 
            edgecolor='black')

# Adicionar valores nas barras
for i, (val0, val1) in enumerate(zip(reclamacao_churn[0], reclamacao_churn[1])):
    axes[5].text(i - width/2, val0 + 50, str(val0), 
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    axes[5].text(i + width/2, val1 + 20, str(val1), 
                ha='center', va='bottom', fontsize=10, fontweight='bold')

axes[5].set_xlabel('Reclamação', fontsize=10)
axes[5].set_ylabel('Quantidade de Clientes', fontsize=10)
axes[5].set_title('Distribuição: Reclamações vs Churn', fontsize=11, fontweight='bold')
axes[5].set_xticks(x_pos)
axes[5].set_xticklabels(['Sem Reclamação', 'Com Reclamação'])
axes[5].legend()

plt.suptitle('Variáveis-Chave vs Churn\nE-Commerce Dataset', 
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

# 🧹 Tratamento e Limpeza de Dados

## 📋 Resumo Executivo

Dataset de E-Commerce preparado para modelagem preditiva de churn através de remoção de duplicatas, imputação de valores nulos e análise de outliers.

**Resultado:** Dataset limpo com 5.074 linhas, 19 colunas, 0 valores nulos, 0 duplicatas.

---

## 🔄 Etapa 1: Remoção de Duplicatas

**Localização:** Cell 6

**Diagnóstico:** 556 linhas duplicadas (9.9% do dataset)

**Método:**
```python
df = df.drop_duplicates()
```

**Resultado:** 5.630 → 5.074 linhas (-556)

**Justificativa:** Eliminar data leakage e preservar independência entre observações.

---

## 💧 Etapa 2: Tratamento de Valores Nulos

**Localização:** Cell 7

### Valores Nulos Identificados:

| Coluna | Nulos | % |
|--------|-------|---|
| Tenure | 231 | 4.6% |
| WarehouseToHome | 221 | 4.4% |
| HourSpendOnApp | 230 | 4.5% |
| DaySinceLastOrder | 288 | 5.7% |
| OrderAmountHikeFromlastYear | 252 | 5.0% |
| CouponUsed | 210 | 4.1% |
| OrderCount | 243 | 4.8% |

### Estratégias Aplicadas:

**MEDIANA** (5 variáveis - distribuições assimétricas ou discretas):
```python
df['Tenure'] = df['Tenure'].fillna(df['Tenure'].median())
df['DaySinceLastOrder'] = df['DaySinceLastOrder'].fillna(df['DaySinceLastOrder'].median())
df['WarehouseToHome'] = df['WarehouseToHome'].fillna(df['WarehouseToHome'].median())
df['CouponUsed'] = df['CouponUsed'].fillna(df['CouponUsed'].median())
df['OrderCount'] = df['OrderCount'].fillna(df['OrderCount'].median())
```

**MÉDIA** (2 variáveis - distribuições simétricas):
```python
df['HourSpendOnApp'] = df['HourSpendOnApp'].fillna(df['HourSpendOnApp'].mean())
df['OrderAmountHikeFromlastYear'] = df['OrderAmountHikeFromlastYear'].fillna(df['OrderAmountHikeFromlastYear'].mean())
```

**Critério de Escolha:**
* **Mediana:** Robusta contra outliers, ideal para distribuições assimétricas e variáveis discretas
* **Média:** Apropriada para distribuições simétricas sem outliers

---

## 📊 Resultado Final

| Métrica | Valor |
|---------|-------|
| Registros | 5.074 |
| Colunas | 19 |
| Valores nulos | 0 ✅ |
| Duplicatas | 0 ✅ |
| Status | Pronto para ML |

---

**📅 Data:** Julho 2026  
**✅ Status:** Dataset limpo e validado


### Fase 3: Feature Engineering (Coluna Calculada)


 ⚠️ Realizei uma análise end-to-end e o resultado foi que os dados preditivos foram melhores deixando só a coluna CashbackAmount do que com a nova coluna cashback_por_pedido. Após confrontar os resultados o veredicto é que cashback_por_pedido não agrega valor preditivo:

### **CashbackAmount original É MAIS CONFIÁVEL** ✅

A análise empírica comprovou que adicionar a feature derivada **piora o modelo**:

| Aspecto | CashbackAmount (original) | cashback_por_pedido (derivada) |
|---------|---------------------------|--------------------------------|
| **Correlação com Churn** | -0.15 (moderada) | **-0.06** (muito fraca) |
| **ROC-AUC do modelo** | **0.9362** | 0.9069 (-2.93 p.p.) |
| **Recall** | **60.12%** | 53.57% (-6.55 p.p.) |

### **Razão Técnica:**

```python
cashback_por_pedido = CashbackAmount / OrderCount
                            ↑                ↑
                   Correlação -0.15   Correlação ~0 (irrelevante)
```

**Dividir uma variável útil por uma irrelevante = diluir o sinal preditivo.**

➡️ **Conclusão:** Continuarei com `CashbackAmount` original no modelo final. 

### Fase 4: Separação, Balanceamento e Escalonamento Seguro

In [0]:
# ==================== ENCODING DE VARIÁVEIS CATEGÓRICAS ====================
print("\n" + "="*70)
print("PASSO 1: ENCODING DE VARIÁVEIS CATEGÓRICAS")
print("="*70)

# Identificar colunas categóricas
cols_categoricas = df.select_dtypes(include=['object']).columns.tolist()
print(f"\nColunas categóricas identificadas: {len(cols_categoricas)}")
for col in cols_categoricas:
    print(f"  • {col}: {df[col].nunique()} categorias únicas")

# Aplicar One-Hot Encoding (todas são nominais, sem ordem intrínseca)
print("\n" + "-"*70)
print("Aplicando One-Hot Encoding...")
print("-"*70)

df_encoded = pd.get_dummies(df, columns=cols_categoricas, drop_first=True)

print(f"\n✅ Encoding concluído!")
print(f"   Colunas ANTES do encoding: {len(df.columns)}")
print(f"   Colunas APÓS o encoding: {len(df_encoded.columns)}")
print(f"   Novas colunas criadas: {len(df_encoded.columns) - len(df.columns)}")

# Exibir primeiras colunas criadas
novas_colunas = [col for col in df_encoded.columns if col not in df.columns]
print(f"\n📋 Exemplos de colunas dummies criadas (primeiras 10):")
for col in novas_colunas[:10]:
    print(f"   • {col}")

print("="*70)

In [0]:
# ==================== SPLIT TREINO/TESTE ESTRATIFICADO ====================
print("\n" + "="*70)
print("PASSO 2: SPLIT TREINO/TESTE ESTRATIFICADO")
print("="*70)

# Separar variável target e preditoras
y = df_encoded['Churn']
X = df_encoded.drop('Churn', axis=1)

print(f"\n🎯 Target (y): Churn")
print(f"   Classe 0 (Retido): {(y==0).sum()} ({(y==0).mean():.2%})")
print(f"   Classe 1 (Churn):  {(y==1).sum()} ({(y==1).mean():.2%})")

print(f"\n📊 Features (X): {X.shape[1]} variáveis preditoras")

# Split estratificado (80% treino, 20% teste)
print("\n" + "-"*70)
print("Aplicando train_test_split (80/20, stratified)...")
print("-"*70)

RANDOM_STATE = 42  # Fixo para reprodutibilidade

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=RANDOM_STATE, 
    stratify=y
)

print(f"\n✅ Split concluído!")
print(f"\n📏 Shapes:")
print(f"   X_train: {X_train.shape}")
print(f"   X_test:  {X_test.shape}")
print(f"   y_train: {y_train.shape}")
print(f"   y_test:  {y_test.shape}")

print(f"\n📈 Proporção de classes (verificação de estratificação):")
print(f"   Original:  Churn={y.mean():.2%}")
print(f"   Treino:    Churn={y_train.mean():.2%}")
print(f"   Teste:     Churn={y_test.mean():.2%}")

if abs(y_train.mean() - y_test.mean()) < 0.01:
    print(f"\n✅ Estratificação bem-sucedida! Proporções equivalentes.")
else:
    print(f"\n⚠️ ATENÇÃO: Diferença de proporção maior que 1%.")

print("="*70)

In [0]:
# ==================== BALANCEAMENTO COM SMOTE (APENAS TREINO) ====================

print("\n" + "="*70)
print("PASSO 3: BALANCEAMENTO COM SMOTE (APENAS CONJUNTO DE TREINO)")
print("="*70)

# Verificar desbalanceamento ANTES
print(f"\n📉 Proporção de classes ANTES do balanceamento (treino):")
print(f"   Classe 0 (Retido): {(y_train==0).sum()} ({(y_train==0).mean():.2%})")
print(f"   Classe 1 (Churn):  {(y_train==1).sum()} ({(y_train==1).mean():.2%})")
print(f"   Ratio: {(y_train==0).sum() / (y_train==1).sum():.2f}:1")

# Aplicar SMOTE
print("\n" + "-"*70)
print("Aplicando SMOTE apenas no conjunto de TREINO...")
print("Conjunto de TESTE permanece inalterado")
print("-"*70)

smote = SMOTE(random_state=RANDOM_STATE)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"\n✅ Balanceamento concluído!")

# Verificar proporção APÓS
print(f"\n📈 Proporção de classes APÓS o balanceamento (treino):")
print(f"   Classe 0 (Retido): {(y_train_balanced==0).sum()} ({(y_train_balanced==0).mean():.2%})")
print(f"   Classe 1 (Churn):  {(y_train_balanced==1).sum()} ({(y_train_balanced==1).mean():.2%})")
print(f"   Ratio: 1:1 (perfeitamente balanceado)")

print(f"\n📏 Shapes APÓS balanceamento:")
print(f"   X_train_balanced: {X_train_balanced.shape} (+{X_train_balanced.shape[0] - X_train.shape[0]} amostras sintéticas)")
print(f"   y_train_balanced: {y_train_balanced.shape}")

print("="*70)

In [0]:
# ==================== IDENTIFICAÇÃO DE VARIÁVEIS CONTÍNUAS ====================
print("\n" + "="*70)
print("PASSO 4: IDENTIFICAÇÃO DE VARIÁVEIS CONTÍNUAS")
print("="*70)

# Identificar colunas contínuas (numéricas originais, antes do encoding)
# Excluir dummies (valores 0/1) criadas pelo One-Hot Encoding
cols_continuas = [
    'Tenure', 'CityTier', 'WarehouseToHome', 'HourSpendOnApp',
    'NumberOfDeviceRegistered', 'SatisfactionScore', 'NumberOfAddress',
    'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount',
    'DaySinceLastOrder', 'CashbackAmount'
]

# Verificar quais existem no dataset encodado
cols_continuas_disponiveis = [col for col in cols_continuas if col in X_train_balanced.columns]

print(f"\n📊 Variáveis contínuas identificadas: {len(cols_continuas_disponiveis)}")
for col in cols_continuas_disponiveis:
    print(f"   • {col}")

# Identificar colunas binárias/dummies (não precisam de escalonamento)
cols_binarias = [col for col in X_train_balanced.columns if col not in cols_continuas_disponiveis]

print(f"\n🎯 Variáveis binárias/dummies (não serão escalonadas): {len(cols_binarias)}")
print(f"   Exemplos: {cols_binarias[:5]}")

print("="*70)

In [0]:
# ==================== ESCALONAMENTO PARA KNN (STANDARDSCALER) ====================

print("\n" + "="*70)
print("PASSO 5: ESCALONAMENTO PARA KNN (StandardScaler)")
print("="*70)

print(f"\n🎯 Objetivo: Preparar dados para K-Nearest Neighbors")

# Criar cópias dos conjuntos
X_train_knn = X_train_balanced.copy()
X_test_knn = X_test.copy()

# Aplicar StandardScaler APENAS nas colunas contínuas
print(f"\n" + "-"*70)
print(f"Aplicando StandardScaler nas {len(cols_continuas_disponiveis)} variáveis contínuas...")
print("-"*70)

scaler_knn = StandardScaler()

# FIT no treino balanceado, TRANSFORM em ambos
X_train_knn[cols_continuas_disponiveis] = scaler_knn.fit_transform(
    X_train_balanced[cols_continuas_disponiveis]
)

X_test_knn[cols_continuas_disponiveis] = scaler_knn.transform(
    X_test[cols_continuas_disponiveis]
)

print(f"\n✅ Escalonamento concluído (fluxo KNN)!")

print(f"\n📋 Colunas escalonadas ({len(cols_continuas_disponiveis)}):")
for col in cols_continuas_disponiveis:
    media_treino = X_train_knn[col].mean()
    std_treino = X_train_knn[col].std()
    print(f"   • {col:<30s} -> Média: {media_treino:+.2f}, Std: {std_treino:.2f}")

print(f"\n📏 Shapes finais (KNN):")
print(f"   X_train_knn: {X_train_knn.shape}")
print(f"   X_test_knn:  {X_test_knn.shape}")

print("="*70)

In [0]:
# ==================== FLUXO PARA ÁRVORE DE DECISÃO (SEM ESCALONAMENTO) ====================
print("\n" + "="*70)
print("PASSO 6: FLUXO PARA ÁRVORE DE DECISÃO (SEM ESCALONAMENTO)")
print("="*70)

print(f"\n🎯 Objetivo: Preparar dados para Decision Tree")

# Usar dados balanceados sem escalonamento
X_train_tree = X_train_balanced.copy()
X_test_tree = X_test.copy()

print(f"\n✅ Fluxo Árvore preparado (sem transformações de escala)!")

print(f"\n📏 Shapes finais (Árvore):")
print(f"   X_train_tree: {X_train_tree.shape}")
print(f"   X_test_tree:  {X_test_tree.shape}")

print(f"\nℹ️ Diferença entre fluxos:")
print(f"   KNN:    StandardScaler aplicado (variáveis contínuas)")
print(f"   Árvore: Dados originais balanceados (sem escalonamento)")

print("="*70)

In [0]:
# ==================== RESUMO FINAL DO PIPELINE ====================
print("\n" + "="*70)
print("🎯 RESUMO FINAL: PIPELINE DE PRÉ-PROCESSAMENTO CONCLUÍDO")
print("="*70)

print(f"\n" + "#"*70)
print("# 1. ENCODING")
print("#"*70)
print(f"   • One-Hot Encoding aplicado em {len(cols_categoricas)} variáveis categóricas")
print(f"   • Colunas finais: {len(df_encoded.columns)} (original: {len(df.columns)})")

print(f"\n" + "#"*70)
print("# 2. SPLIT ESTRATIFICADO")
print("#"*70)
print(f"   • test_size: 0.20")
print(f"   • stratify: sim (proporção de classes mantida)")
print(f"   • random_state: {RANDOM_STATE}")
print(f"   • Shapes:")
print(f"      - Treino: {X_train.shape}")
print(f"      - Teste:  {X_test.shape}")

print(f"\n" + "#"*70)
print("# 3. BALANCEAMENTO COM SMOTE(APENAS TREINO)")
print("#"*70)
print(f"   • Proporção ANTES:  Churn={y_train.mean():.2%}")
print(f"   • Proporção DEPÓIS: Churn={y_train_balanced.mean():.2%} (balanceado 1:1)")
print(f"   • Amostras adicionadas: {X_train_balanced.shape[0] - X_train.shape[0]}")
print(f"   • Conjunto de TESTE: INALTERADO (realista)")

print(f"\n" + "#"*70)
print("# 4. ESCALONAMENTO")
print("#"*70)

print(f"\n   🤖 FLUXO KNN (COM StandardScaler):")
print(f"      • Colunas escalonadas: {len(cols_continuas_disponiveis)} variáveis contínuas")
print(f"      • Scaler FIT no treino balanceado")
print(f"      • Scaler TRANSFORM no teste")
print(f"      • Dummies (0/1) NÃO escalonadas")

print(f"\n   🌳 FLUXO ÁRVORE (SEM StandardScaler):")
print(f"      • Razão: Árvores fazem cortes monotônicos, não dependem de escala")
print(f"      • Nenhuma transformação aplicada")

print(f"\n" + "#"*70)
print("# 5. DATASETS FINAIS DISPONÍVEIS")
print("#"*70)

print(f"\n   📦 Para KNN:")
print(f"      - X_train_knn: {X_train_knn.shape}")
print(f"      - X_test_knn:  {X_test_knn.shape}")
print(f"      - y_train_balanced: {y_train_balanced.shape}")
print(f"      - y_test: {y_test.shape}")

print(f"\n   📦 Para Árvore de Decisão:")
print(f"      - X_train_tree: {X_train_tree.shape}")
print(f"      - X_test_tree:  {X_test_tree.shape}")
print(f"      - y_train_balanced: {y_train_balanced.shape}")
print(f"      - y_test: {y_test.shape}")

print(f"\n" + "="*70)
print("✅ PIPELINE DE PRÉ-PROCESSAMENTO 100% CONCLUÍDO!")
print("="*70)
print(f"\n🚀 PRÓXIMO PASSO: Treinamento e avaliação dos modelos")
print("="*70)

### Fase 5: Modelagem e Validação (O Desafio do Overfitting)